# CertGen Kaggle environment diagnostic — T4 ×2

Select **GPU T4 ×2**. This is `synthetic_validation_only`, `not_empirical_evidence`, and `claim_allowed=false`. Run top-to-bottom. Do not enable model assets for this diagnostic.


## 0 Human instructions

Select `GPU T4 ×2`; the parent process must not import or initialize CUDA.


## 1 Immutable user configuration


In [ ]:
from __future__ import annotations
import json, multiprocessing as mp, os, subprocess
from pathlib import Path
mp.set_start_method("spawn", force=True)
from certgen.notebooks.kaggle_io import load_frozen_configuration, safe_extract_one_input_package, verify_input_integrity
INPUT_ROOT = safe_extract_one_input_package()
verify_input_integrity(INPUT_ROOT, ignored={".source_sha256"})
CONFIG = load_frozen_configuration(INPUT_ROOT)
MODE = CONFIG["mode"]
WORK_ROOT = Path("/kaggle/working/certgen-diagnostic")
RUN_ROOT = WORK_ROOT / CONFIG["run_id"]
RUN_ROOT.mkdir(parents=True, exist_ok=True)


## 2 Input discovery


In [ ]:
print(json.dumps({"input_root": str(INPUT_ROOT), "configuration_hash": CONFIG["configuration_hash"]}, indent=2))


## 3 Environment diagnostics


In [ ]:
probe = subprocess.run(["nvidia-smi", "-L"], check=True, capture_output=True, text=True)
GPU_LINES = [line for line in probe.stdout.splitlines() if line.strip().startswith("GPU ")]
if len(GPU_LINES) != 2: raise RuntimeError(f"GPU T4 x2 required; found {len(GPU_LINES)} devices")


## 4 Dependency setup and validation

The bootstrap runs `python -m pip check` and writes `dependency_report.json`, `dependency_freeze.txt`, and `pip_check.txt`.


In [ ]:
from certgen.notebooks.environment_bootstrap import bootstrap_environment
DEPENDENCIES = bootstrap_environment("kaggle_t4x2_preflight", output_dir=RUN_ROOT / "dependencies", network_allowed=CONFIG["dependency_network_allowed"], apply=True, install_mode=CONFIG["dependency_mode"])
if DEPENDENCIES["status"] != "ENVIRONMENT_COMPATIBLE" or DEPENDENCIES["pip_check"]["returncode"] != 0: raise RuntimeError("dependency validation failed")


## 5 Asset discovery and validation


In [ ]:
from certgen.notebooks.model_assets import AssetPolicy
ASSET_POLICY = AssetPolicy(CONFIG["asset_policy"])
ASSET_VALIDATION = {"required_assets": [], "passed": True, "claim_allowed": False}


## 6 Configuration/provenance validation


In [ ]:
from certgen.cvpr.contracts import atomic_write_json
PROVENANCE = {"configuration_hash": CONFIG["configuration_hash"], "input_manifest_hash": CONFIG["input_manifest_hash"], "claim_allowed": False}
atomic_write_json(PROVENANCE, RUN_ROOT / "provenance.json")


## 7 Tiny dual-GPU dry run


In [ ]:
import multiprocessing as mp
mp.set_start_method("spawn", force=True)
from certgen.notebooks.subprocess_orchestrator import WorkerSpec, run_workers
TINY_SPECS = [
    WorkerSpec(
        worker_id=f"tiny_gpu_{gpu}",
        module="certgen.notebooks.workers.diagnostic_worker",
        physical_gpu=gpu,
        shard_id=f"tiny_gpu_{gpu}",
        args=("--out", str(RUN_ROOT / "tiny_gpu_diagnostic" / f"gpu_{gpu}"),
              "--configuration-hash", CONFIG["configuration_hash"],
              "--input-manifest-hash", str(CONFIG.get("input_manifest_hash", CONFIG.get("reference_manifest_hash", "diagnostic_static_input")))),
        completion_marker=str(RUN_ROOT / "tiny_gpu_diagnostic" / f"gpu_{gpu}" / "worker_completion.json"),
        configuration_hash=CONFIG["configuration_hash"],
        input_manifest_hash=str(CONFIG.get("input_manifest_hash", CONFIG.get("reference_manifest_hash", "diagnostic_static_input"))),
        asset_manifest_hash="no_assets_required",
        worker_type="diagnostic",
        config_schema_version="certgen.kaggle.diagnostic_config.v1",
        output_schema_version="certgen.kaggle.diagnostic_output.v1",
    )
    for gpu in range(2)
]
TINY_DUAL_GPU = run_workers(TINY_SPECS, output_dir=RUN_ROOT / "tiny_gpu_orchestration", resume=MODE == "resume")
if sorted(row["physical_gpu"] for row in TINY_DUAL_GPU["workers"]) != [0, 1]:
    raise RuntimeError("tiny dry run did not execute one worker on each physical GPU")


## 8 Runtime calibration


In [ ]:
from certgen.cvpr.contracts import atomic_write_json
CALIBRATION_ROWS = [
    json.loads((RUN_ROOT / "tiny_gpu_diagnostic" / f"gpu_{gpu}" / "diagnostic_report.json").read_text(encoding="utf-8"))
    for gpu in range(2)
]
RUNTIME_CALIBRATION = {
    "model_load_seconds": [row["model_load_seconds"] for row in CALIBRATION_ROWS],
    "warmup_seconds": [row["warmup_seconds"] for row in CALIBRATION_ROWS],
    "throughput_iterations_per_second": [row["throughput_iterations_per_second"] for row in CALIBRATION_ROWS],
    "peak_vram_bytes": [row["peak_allocated_bytes"] for row in CALIBRATION_ROWS],
    "safe_batch_size": min(row["safe_batch_size"] for row in CALIBRATION_ROWS),
    "planning_only": True,
    "not_empirical_evidence": True,
    "claim_allowed": False,
}
atomic_write_json(RUNTIME_CALIBRATION, RUN_ROOT / "runtime_calibration.json")


## 9 Full parallel execution


In [ ]:
# For the diagnostic stage, the two-worker dry run is the complete parallel execution.
ORCHESTRATION = TINY_DUAL_GPU


## 10 Merge and validation


In [ ]:
from certgen.notebooks.kaggle_io import all_worker_statuses_complete, write_integrity_manifest
if not all_worker_statuses_complete(ORCHESTRATION): raise RuntimeError("dual-GPU diagnostic worker failure")
STATUS = {"status_code": "KAGGLE_DIAGNOSTIC_PASS", "gpu_count": 2, "configuration_hash": CONFIG["configuration_hash"], "synthetic_validation_only": True, "not_empirical_evidence": True, "claim_allowed": False}
atomic_write_json(STATUS, RUN_ROOT / "diagnostic_status.json")
write_integrity_manifest(RUN_ROOT)


## 11 Atomic output ZIP


In [ ]:
from certgen.notebooks.final_zip import finalize_output_zip, validate_final_zip, write_multipart_fallback
ZIP_PATH = Path("/kaggle/working/certgen_kaggle_environment_diagnostic_output.zip")
ZIP = finalize_output_zip(RUN_ROOT, ZIP_PATH, mode=MODE, configuration_hash=CONFIG["configuration_hash"], asset_manifest_hash="no_assets_required")
if not validate_final_zip(RUN_ROOT, ZIP_PATH)["passed"]: raise RuntimeError("final diagnostic ZIP revalidation failed")
MULTIPART = write_multipart_fallback(ZIP_PATH) if ZIP_PATH.stat().st_size > 3800 * 1024**2 else None


## 12 Local handoff


Download `certgen_kaggle_environment_diagnostic_output.zip` to `data/kaggle_returns/diagnostic/` and run exactly:

`CUDA_VISIBLE_DEVICES="" CERTGEN_CPU_ONLY=1 python3 scripts/run_all_available_cpu_stages.py --resume --explain`

Preserve worker logs and status files on failure; resume only when configuration and input hashes are unchanged.
